# 04 — Churn Model

**Purpose:** Train and evaluate the Logistic Regression churn-prediction pipeline,
produce all required metrics and evaluation plots, and persist model artifacts.

**Input:** `data/processed/customer_features.csv` — pre-computed eligible-customer
feature table (1,765 rows) produced by notebook 03.  The Excel workbook is NOT
reloaded here.

**Critical design notes:**
- `Churn_90D` is a project-defined future-inactivity proxy; snapshot = 2011-08-31;
  future window = 2011-09-01 to 2011-11-29.
- RFM scores (`R`, `F`, `M`, `RFM_Score`, `RFM_Segment`) are **descriptive only**;
  they are **NOT** used as model predictors.
- `CustomerID` is **NOT** a predictor.
- All preprocessing (StandardScaler) is inside the Pipeline and fitted **only** on
  training data — no pre-split scaling.
- The test set is held out until final evaluation; no test statistics leak into the
  scaler or any other preprocessing step.


## 1 — Imports and Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import json
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.models.train import (
    NUMERIC_FEATURES,
    build_pipeline,
    load_pipeline,
    save_pipeline,
    split_data,
    train,
)
from src.models.evaluate import (
    compute_metrics,
    get_feature_importance,
    plot_confusion_matrix,
    plot_precision_recall_curve,
    plot_roc_curve,
    save_evaluation_json,
    threshold_analysis,
)
from src.config import (
    CUSTOMER_FEATURES_PATH,
    MODEL_ARTIFACT_PATH,
    PROCESSED_DIR,
    REPORTS_DIR,
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

Setup complete.


## 2 — Load Customer Feature Table

In [2]:
df = pd.read_csv(CUSTOMER_FEATURES_PATH, index_col='CustomerID')
print(f'Loaded {len(df):,} rows from {CUSTOMER_FEATURES_PATH.name}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Loaded 1,765 rows from customer_features.csv
Columns: ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'TotalItems', 'ActiveMonths', 'TenureDays', 'PurchaseSpanDays', 'Country', 'R', 'F', 'M', 'RFM_Score', 'RFM_Segment', 'Churn_90D']


,Recency,Frequency,Monetary,AvgOrderValue,UniqueProducts,TotalItems,ActiveMonths,TenureDays,PurchaseSpanDays,Country,R,F,M,RFM_Score,RFM_Segment,Churn_90D
CustomerID,,,,,,,,,,,,,,,,
12347.0,29,5,2790.86,558.172000,82,1590,5,267,238,Iceland,3,4,5,12,High Value,0
12348.0,148,3,1487.24,495.746667,22,2124,3,258,110,Finland,1,2,4,7,Medium Value,0
12352.0,162,5,1561.81,312.362000,26,254,2,196,34,Norway,1,4,4,9,Medium Value,0


## 3 — Validate Target (Churn_90D)

In [3]:
assert 'Churn_90D' in df.columns, 'Missing target column Churn_90D'
counts = df['Churn_90D'].value_counts().sort_index()
total = len(df)
print('Churn_90D distribution:')
for label, cnt in counts.items():
    print(f'  {label} : {cnt:,}  ({cnt/total:.2%})')
assert set(df['Churn_90D'].unique()) == {0, 1}, 'Unexpected target values'

Churn_90D distribution:
  0 : 1,275  (72.24%)
  1 : 490  (27.76%)


## 4 — Select Predictors

**Numeric predictors only** (primary model).  
Excluded from X: `CustomerID`, `Churn_90D`, `R`, `F`, `M`, `RFM_Score`, `RFM_Segment`, `Country`.

The RFM quintile scores are **descriptive** segmentation outputs and must not be used as model inputs.

In [4]:
print('Primary numeric predictors:', NUMERIC_FEATURES)

# Sanity: all NUMERIC_FEATURES are present in df
missing = [f for f in NUMERIC_FEATURES if f not in df.columns]
assert not missing, f'Missing feature columns: {missing}'

features = df[NUMERIC_FEATURES].copy()
labels = df[['Churn_90D']].copy()
print(f'Feature matrix shape: {features.shape}')
print(f'Null values in features: {features.isnull().sum().sum()}')

Primary numeric predictors: ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'TotalItems', 'ActiveMonths', 'TenureDays', 'PurchaseSpanDays']
Feature matrix shape: (1765, 9)
Null values in features: 0


## 5 — Leakage Checks

Verify that no forbidden predictor is included and that no future-window data
has leaked into the feature set.

In [5]:
FORBIDDEN_PREDICTORS = {'CustomerID', 'Churn_90D', 'R', 'F', 'M',
                        'RFM_Score', 'RFM_Segment'}

feature_set = set(NUMERIC_FEATURES)
leaked = feature_set & FORBIDDEN_PREDICTORS
assert not leaked, f'LEAKAGE — forbidden predictor(s) in X: {leaked}'

assert 'Churn_90D' not in features.columns, 'LEAKAGE: Churn_90D found in X'

# CustomerID must not appear as a column (it is the index only)
assert 'CustomerID' not in features.columns, 'LEAKAGE: CustomerID found in X columns'

print('All leakage checks passed.')
print(f'  Forbidden predictors checked: {sorted(FORBIDDEN_PREDICTORS)}')
print(f'  Feature set: {sorted(feature_set)}')

All leakage checks passed.
  Forbidden predictors checked: ['Churn_90D', 'CustomerID', 'F', 'M', 'R', 'RFM_Score', 'RFM_Segment']
  Feature set: ['ActiveMonths', 'AvgOrderValue', 'Frequency', 'Monetary', 'PurchaseSpanDays', 'Recency', 'TenureDays', 'TotalItems', 'UniqueProducts']


## 6 — Train / Test Split (80 / 20, Stratified)

In [6]:
X_train, X_test, y_train, y_test = split_data(features, labels)
print(f'Train : {len(X_train):,} rows   churn rate = {y_train.mean():.4f}')
print(f'Test  : {len(X_test):,} rows   churn rate = {y_test.mean():.4f}')
print(f'Train class counts: {y_train.value_counts().sort_index().to_dict()}')
print(f'Test  class counts: {y_test.value_counts().sort_index().to_dict()}')

# Verify no customer appears in both splits
overlap = set(X_train.index) & set(X_test.index)
assert not overlap, f'Customer-level leakage: {len(overlap)} customers in both splits'

Train : 1,412 rows   churn rate = 0.2776
Test  : 353 rows   churn rate = 0.2776
Train class counts: {0: 1020, 1: 392}
Test  class counts: {0: 255, 1: 98}


## 7 — Pipeline Definition

The Pipeline contains:
1. `StandardScaler` — fitted **only on training data** inside the pipeline.  
2. `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`.

In [7]:
pipeline = build_pipeline()
print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


## 8 — Model Training

In [8]:
pipeline = train(X_train, y_train, pipeline)
print('Model trained successfully.')

Model trained successfully.


## 9 — Prediction on Test Set

In [9]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]   # Churn_Probability (class 1)

print(f'Predicted classes  — unique values: {sorted(set(y_pred))}')
print(f'Probabilities range: [{y_prob.min():.4f}, {y_prob.max():.4f}]')
print(f'Mean predicted probability: {y_prob.mean():.4f}')

Predicted classes  — unique values: [np.int64(0), np.int64(1)]
Probabilities range: [0.0000, 0.8109]
Mean predicted probability: 0.4539


## 10 — Classification Metrics

In [10]:
metrics = compute_metrics(pipeline, X_test, y_test)
print('--- Classification Metrics (default threshold = 0.50) ---')
for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')

--- Classification Metrics (default threshold = 0.50) ---
  accuracy    : 0.5864
  precision   : 0.3723
  recall      : 0.7143
  f1          : 0.4895
  roc_auc     : 0.7063
  pr_auc      : 0.4686


## 11 — ROC-AUC and PR-AUC

In [11]:
print(f"ROC-AUC : {metrics['roc_auc']:.4f}")
print(f"PR-AUC  : {metrics['pr_auc']:.4f}")
print()
print('Note: because the target is imbalanced (~72% active, ~28% churned),')
print('PR-AUC is the more informative indicator of classifier quality.')

ROC-AUC : 0.7063
PR-AUC  : 0.4686

Note: because the target is imbalanced (~72% active, ~28% churned),
PR-AUC is the more informative indicator of classifier quality.


## 12 — Confusion Matrix

In [12]:
fig_cm = plot_confusion_matrix(
    pipeline, X_test, y_test,
    save_path=REPORTS_DIR / 'confusion_matrix.png'
)
print(f'Saved: {REPORTS_DIR / "confusion_matrix.png"}')

Saved: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\confusion_matrix.png


## 13 — ROC Curve

In [13]:
fig_roc = plot_roc_curve(
    pipeline, X_test, y_test,
    save_path=REPORTS_DIR / 'roc_curve.png'
)
print(f'Saved: {REPORTS_DIR / "roc_curve.png"}')

Saved: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\roc_curve.png


## 14 — Precision-Recall Curve

In [14]:
fig_pr = plot_precision_recall_curve(
    pipeline, X_test, y_test,
    save_path=REPORTS_DIR / 'precision_recall_curve.png'
)
print(f'Saved: {REPORTS_DIR / "precision_recall_curve.png"}')

Saved: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\precision_recall_curve.png


## 15 — Threshold Analysis

The default threshold is **0.50**.  The table below reports Precision, Recall, and F1
at a range of candidate thresholds.  **No threshold is silently adopted** — the
application uses the continuous `Churn_Probability` for risk tiers.

In [15]:
thresh_df = threshold_analysis(pipeline, X_test, y_test)
print(thresh_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print()
print('Trade-off note: lower thresholds increase Recall (catch more churners)')
print('at the cost of Precision (more false alarms).  The optimal operating')
print('threshold depends on the business cost of missed churners vs. false alarms.')

 threshold  precision  recall     f1
    0.2000     0.3297  0.9286 0.4866
    0.2500     0.3308  0.8878 0.4820
    0.3000     0.3386  0.8776 0.4886
    0.3500     0.3529  0.8571 0.5000
    0.4000     0.3511  0.8061 0.4892
    0.4500     0.3623  0.7653 0.4918
    0.5000     0.3723  0.7143 0.4895
    0.5500     0.4061  0.6837 0.5095
    0.6000     0.4385  0.5816 0.5000

Trade-off note: lower thresholds increase Recall (catch more churners)
at the cost of Precision (more false alarms).  The optimal operating
threshold depends on the business cost of missed churners vs. false alarms.


## 16 — Coefficient Interpretation

Coefficients reflect standardised inputs.  Positive coefficients are
**associated with higher predicted inactivity risk**; negative coefficients
are associated with lower predicted inactivity risk.  These are associations
only — no causal claims are made.

In [16]:
coef_df = get_feature_importance(pipeline)
print('Feature coefficient table (sorted by |coefficient|):')
print(coef_df.to_string(index=False))
print()
pos = coef_df[coef_df['coefficient'] > 0][['feature', 'coefficient']].head(3)
neg = coef_df[coef_df['coefficient'] < 0][['feature', 'coefficient']].head(3)
print('Strongest positive associations (higher value → higher inactivity risk):')
print(pos.to_string(index=False))
print('\nStrongest negative associations (higher value → lower inactivity risk):')
print(neg.to_string(index=False))

Feature coefficient table (sorted by |coefficient|):
         feature  coefficient  abs_coefficient
    ActiveMonths    -0.789998         0.789998
       Frequency    -0.774142         0.774142
      TotalItems    -0.768869         0.768869
  UniqueProducts    -0.410150         0.410150
        Monetary     0.374441         0.374441
         Recency     0.131990         0.131990
      TenureDays     0.122682         0.122682
PurchaseSpanDays     0.006174         0.006174
   AvgOrderValue    -0.002792         0.002792

Strongest positive associations (higher value → higher inactivity risk):
   feature  coefficient
  Monetary     0.374441
   Recency     0.131990
TenureDays     0.122682

Strongest negative associations (higher value → lower inactivity risk):
     feature  coefficient
ActiveMonths    -0.789998
   Frequency    -0.774142
  TotalItems    -0.768869


## 17 — Generate Test Predictions (Churn_Probability)

In [17]:
test_preds = X_test[NUMERIC_FEATURES].copy()
test_preds['Churn_90D'] = y_test
test_preds['Churn_Probability'] = y_prob
test_preds['Predicted_Class'] = y_pred

# Reset index so CustomerID becomes a column
test_preds = test_preds.reset_index()

out_cols = ['CustomerID', 'Churn_90D', 'Churn_Probability', 'Predicted_Class'] + NUMERIC_FEATURES
test_preds = test_preds[out_cols]

pred_path = PROCESSED_DIR / 'test_predictions.csv'
test_preds.to_csv(pred_path, index=False)
print(f'Saved {len(test_preds):,} test predictions to {pred_path}')
test_preds[['CustomerID', 'Churn_90D', 'Churn_Probability', 'Predicted_Class']].head(5)

Saved 353 test predictions to D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\data\processed\test_predictions.csv


,CustomerID,Churn_90D,Churn_Probability,Predicted_Class
0,14414.0,0,0.435282,0
1,16743.0,0,0.233270,0
2,13538.0,0,0.354721,0
3,18260.0,1,0.197729,0
4,12749.0,0,0.513115,1


## 18 — Save Model Artifact

In [18]:
artifact_path = save_pipeline(pipeline)
print(f'Model artifact saved to: {artifact_path}')

Model artifact saved to: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\models\churn_model.joblib


## 19 — Save Evaluation Outputs (JSON)

In [19]:
eval_path = REPORTS_DIR / 'model_evaluation.json'
save_evaluation_json(
    metrics=metrics,
    thresh_df=thresh_df,
    coef_df=coef_df,
    y_train=y_train,
    y_test=y_test,
    save_path=eval_path,
    default_threshold=0.50,
)
print(f'Evaluation JSON saved to: {eval_path}')

# Quick sanity check — load back and print keys
with open(eval_path) as f:
    saved = json.load(f)
print('Keys in saved evaluation JSON:', list(saved.keys()))

Evaluation JSON saved to: D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\model_evaluation.json
Keys in saved evaluation JSON: ['train_size', 'test_size', 'class_counts', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'default_threshold', 'threshold_analysis', 'feature_coefficients']


## 20 — Artifact Load Validation

Verify the saved pipeline can be loaded from disk and produce probabilities
on the test feature columns without any refitting.

In [20]:
loaded_pipeline = load_pipeline()
loaded_probs = loaded_pipeline.predict_proba(X_test[NUMERIC_FEATURES])[:, 1]

# Probabilities from disk must match in-memory predictions
import numpy as np
np.testing.assert_allclose(loaded_probs, y_prob, rtol=1e-5)
print(f'Artifact load validation PASSED — loaded pipeline from {MODEL_ARTIFACT_PATH}')
print(f'Probability range from loaded model: [{loaded_probs.min():.4f}, {loaded_probs.max():.4f}]')

Artifact load validation PASSED — loaded pipeline from D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\models\churn_model.joblib
Probability range from loaded model: [0.0000, 0.8109]


## 21 — Summary

All Phase 4 outputs have been produced.  See reports below for a complete summary.

In [21]:
print('=' * 60)
print('PHASE 4 SUMMARY')
print('=' * 60)
print(f'Train size     : {len(X_train):,}')
print(f'Test size      : {len(X_test):,}')
print(f'Train churn    : {y_train.sum():,} / {len(y_train):,} ({y_train.mean():.2%})')
print(f'Test  churn    : {y_test.sum():,} / {len(y_test):,} ({y_test.mean():.2%})')
print()
print('Metrics at default threshold (0.50):')
for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')
print()
print('Artifacts:')
print(f'  {MODEL_ARTIFACT_PATH}')
print(f'  {PROCESSED_DIR / "test_predictions.csv"}')
print(f'  {REPORTS_DIR / "model_evaluation.json"}')
print(f'  {REPORTS_DIR / "confusion_matrix.png"}')
print(f'  {REPORTS_DIR / "roc_curve.png"}')
print(f'  {REPORTS_DIR / "precision_recall_curve.png"}')

PHASE 4 SUMMARY
Train size     : 1,412
Test size      : 353
Train churn    : 392 / 1,412 (27.76%)
Test  churn    : 98 / 353 (27.76%)

Metrics at default threshold (0.50):
  accuracy    : 0.5864
  precision   : 0.3723
  recall      : 0.7143
  f1          : 0.4895
  roc_auc     : 0.7063
  pr_auc      : 0.4686

Artifacts:
  D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\models\churn_model.joblib
  D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\data\processed\test_predictions.csv
  D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\model_evaluation.json
  D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\confusion_matrix.png
  D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\roc_curve.png
  D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\reports\precision_recall_curve.png
